In [ ]:
print('hello')

# Section_1_Basics

## Intracting_with_llm

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="deepseek-r1:8b",
    temperature=0.5,
    num_predict=250  # maximum number of tokens the model is allowed to generate in the response
    )

print("start")
response = llm.invoke("What is the capital of France? in one line")
print(response)

start
content='Paris is the capital city of France.' additional_kwargs={} response_metadata={'model': 'deepseek-r1:8b', 'created_at': '2026-06-02T08:20:14.9384188Z', 'done': True, 'done_reason': 'stop', 'total_duration': 70058251800, 'load_duration': 8977046600, 'prompt_eval_count': 12, 'prompt_eval_duration': 4074722000, 'eval_count': 173, 'eval_duration': 56505557100, 'logprobs': None, 'model_name': 'deepseek-r1:8b', 'model_provider': 'ollama'} id='lc_run--019e876a-2c2e-7e41-aa33-cfd7325f01d8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 12, 'output_tokens': 173, 'total_tokens': 185}


In [ ]:
reply=response.content
print(reply)

## Connecting with LangSmith...

In [ ]:
import os
from dotenv import load_dotenv

load=load_dotenv(".env",override=True) # override=True will override the existing env variables with the ones in the .env file, if there are any conflicts. This is useful when we want to test different env variables without having to change the system env variables.

""""see how to connect in langsmith web interface and see the logs of the conversations"""



## Prompt and chat template 

In [ ]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("What is the capital of {country}?")  # we are using the prompt template to create a prompt that can take a variable as input. Here we are using the variable "country" which will be replaced by the value we pass while invoking the prompt.
prompt= prompt.invoke({"country":"France"})
content=llm.invoke(prompt).content
print(content)

## Chat Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate(
    [("system", "You are a helpful assistant that answers questions about geography.") ,
     ("human", "What is the capital of {country}?")]
)
prompt= prompt.invoke({"country":"France"})
content=llm.invoke(prompt).content
print(content)

## MessagePlaceholder

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage



prompt = ChatPromptTemplate(
    [("system", "You are a helpful assistant that answers questions about geography.") ,
        MessagesPlaceholder("msg")
])
prompt= prompt.invoke({"msg":[HumanMessage("What is the capital of France?")]})
content=llm.invoke(prompt).content
print(content)

## Use stream in place of invoke

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

prompt = ChatPromptTemplate(
    [("system", "You are a helpful assistant that answers questions about geography.") ,
        MessagesPlaceholder("msg")
])
prompt= prompt.invoke({"msg":[HumanMessage("What is the capital of France?")]})
print("start streaming")
for i in llm.stream(prompt):
    content=i.content
    print(content, end="")

# Section_2_Chaining and Runnables

## Understanding Chaining and Runnables

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate(
    [("system", "You are a helpful assistant that answers questions about geography.") ,
     ("human", "What is the capital of {country}?")]
)
parser = StrOutputParser()

chain = prompt | llm | parser   #  both prompts and llm are runnables, so we can use the pipe operator to chain them together. The output of the prompt will be passed as input to the llm.
response = chain.invoke({"country":"France"})

print(response)  # when parser is used the response is a string, otherwise it is a message object. then only use response.content to get the string response.

## chainning multiple chains

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate(
    [("system", "You are a helpful assistant that answers questions about geography.") ,
     ("human", "What is the capital of {country} and about it?")]
)
parser = StrOutputParser()

detailedResponseChain1 = prompt | llm | parser 

headingInfoTemplate = ChatPromptTemplate.from_template("""Analyse the response and get me just the heading 
                                         from the {response}. The heading is the part of the response which 
                                         is in bold and is at the start of the response. Response should
                                         be in the bullet points.""")

chainWithHeadingChain2 = {"response":detailedResponseChain1} | headingInfoTemplate | llm | parser

response = chainWithHeadingChain2.invoke({"country":"France"})
print(response)




## Running chain in Parallel

In [ ]:
from langchain_ollama import ChatOllama

llm1 = ChatOllama(
    base_url="http://localhost:11434",
    model="deepseek-r1:8b",
    temperature=0.5,
    num_predict=250  # maximum number of tokens the model is allowed to generate in the response
    )

print("start")
response = llm.invoke("What is the capital of France? in one line")
print(response)


# when you want to use two different models or llm

llm2 = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen2.5",
    temperature=0.5,
    num_predict=250  # maximum number of tokens the model is allowed to generate in the response
    )

print("start")
response = llm2.invoke("What is the capital of France? in one line")
print(response)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

prompt = ChatPromptTemplate(
    [("system", "You are a helpful assistant that answers questions about geography.") ,
     ("human", "What is the capital of {country} and about it?")]
)
parser = StrOutputParser()

detailedResponseChain1 = prompt | llm1 | parser 

headingInfoTemplate = ChatPromptTemplate.from_template("""Analyse the response and get me just the heading 
                                         from the {response}. The heading is the part of the response which 
                                         is in bold and is at the start of the response. Response should
                                         be in the bullet points.""")

chainWithHeadingChain2 = {"response":detailedResponseChain1} | headingInfoTemplate | llm2 | parser

parrellRunnable = RunnableParallel(chain1=detailedResponseChain1, chain2=chainWithHeadingChain2).invoke({"country":"France"})
response= parrellRunnable.invoke({"country":"France"})

print(response[detailedResponseChain1])  # this will give the response from the first chain which is detailedResponseChain1
print("/n/n") 
print(response[chainWithHeadingChain2])  # this will give the response from the second chain which is chainWithHeadingChain2


## RunnableLambda

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
prompt = ChatPromptTemplate(
    [("system", "You are a helpful assistant that answers questions about geography.") ,
     ("human", "What is the capital of {country} and about it?")]
)
parser = StrOutputParser()

detailedResponseChain1 = prompt | llm | parser 

headingInfoTemplate = ChatPromptTemplate.from_template("""Analyse the response and get me just the heading 
                                         from the {response}. The heading is the part of the response which 
                                         is in bold and is at the start of the response. Response should
                                         be in the bullet points.""")


def choose_llm(response):
    if "heading1" in response:
        return llm1
    else:
        return llm2
    
llm_selector = RunnableLambda(choose_llm)

chainWithHeadingChain2 = {"response":detailedResponseChain1} | headingInfoTemplate | llm_selector | parser

response = chainWithHeadingChain2.invoke({"country":"France"})
print(response)

## Using Chain decorator

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
prompt = ChatPromptTemplate(
    [("system", "You are a helpful assistant that answers questions about geography.") ,
     ("human", "What is the capital of {country} and about it?")]
)
parser = StrOutputParser()

detailedResponseChain1 = prompt | llm | parser 

headingInfoTemplate = ChatPromptTemplate.from_template("""Analyse the response and get me just the heading 
                                         from the {response}. The heading is the part of the response which 
                                         is in bold and is at the start of the response. Response should
                                         be in the bullet points.""")

@chain
def choose_llm(response):
    if "heading1" in response:
        return llm1
    else:
        return llm2
    


chainWithHeadingChain2 = {"response":detailedResponseChain1} | headingInfoTemplate | choose_llm | parser

response = chainWithHeadingChain2.invoke({"country":"France"})
print(response)

# Section_3_ChatHistory

In [ ]:
from langchain_ollama import ChatOllama

llm1 = ChatOllama(
    base_url="http://localhost:11434",
    model="deepseek-r1:8b",
    temperature=0.5,
    num_predict=250  # maximum number of tokens the model is allowed to generate in the response
    )

print("start")
response = llm.invoke("What is the capital of France? in one line")
print(response)


# when you want to use two different models or llm

llm2 = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen2.5",
    temperature=0.5,
    num_predict=250  # maximum number of tokens the model is allowed to generate in the response
    )

print("start")
response = llm2.invoke("What is the capital of France? in one line")
print(response)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


template = ChatPromptTemplate.from_messages([
    ("human", "{prompt}"),
    ("placeholder", "{history}")  # this will create a placeholder for the chat history which will be passed to the llm while invoking the prompt. The chat history will be stored in the ChatMessageHistory object which is a list of messages.
    ])


chain = template | llm | StrOutputParser()


store= {}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]



history= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_message_key="prompt",
    history_message_key="history"
)  # this will create a runnable with message history which will store the chat history in the ChatMessageHistory object.


session_id="Karthik"

get_session_history(session_id).clear

response1= history.invoke({"prompt":"What is the benefit of running LLM in local Machine?"}, 
                         config={"configurable":{"session_id":session_id}})  # this will invoke the history runnable which will get the chat history for the given session id and pass it to the prompt while invoking it. The response from the llm will be stored in the chat history object for that session id.


response2= history.invoke({"prompt":"how about from Cloud?"}, 
                         config={"configurable":{"session_id":session_id}})  # this will invoke the history


print(response1)
print("\n\n")
print(response2)

## ChatMessage History with SqlChatMessageHistory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
